In [1]:
# ! brew install ollama
# ! ollama serve
# ! ollama pull llama3:8b
! ollama list

NAME           ID              SIZE      MODIFIED       
phi3:medium    cf611a26b048    7.9 GB    18 minutes ago    
llama3.1:8b    46e0c10c039e    4.9 GB    38 minutes ago    
llama3:8b      365c0bd3c000    4.7 GB    9 hours ago       


In [2]:
import json
import pandas as pd
from typing import Dict, List, Any
import numpy as np

def extract_gtfs(file_path):
    json_data = Dict[str, Any]
    with open(file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    normalized_data = []
    for poi in json_data['elements']:
        # Copy basic properties like type, id, lat, lon
        item = {k: v for k, v in poi.items() if k != 'tags'}
        if 'tags' in poi and isinstance(poi['tags'], dict):
            item.update(poi['tags'])

        normalized_data.append(item)

    df = pd.DataFrame(normalized_data)
    df = df.replace('nan', np.nan)

    if not df.empty:
        priority_cols = ['id', 'name', 'lat', 'lon', 'opening_hours', 'amenity', 'cuisine', 'wheelchair', 'toilets', 'access']
        existing_cols = [col for col in priority_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]
        df = df[existing_cols + other_cols]

    return df

file_path = './data_collection/raw_data/raw_osm.json'
data_df = extract_gtfs(file_path)

In [3]:
data_df['amenity'].unique()

array([nan, 'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain',
       'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium',
       'conference_centre', 'fire_station', 'cinema', 'toilets',
       'arts_centre', 'place_of_worship'], dtype=object)

## Open model (Llama)

In [4]:
# User-inputted variables
START_LAT = 32.5106
START_LON = -117.0626 
NUM_POIS_TO_VISIT = 4
TIME_PER_POI = 1.5
MAX_DIST = 15

# User Preferences
USER_PREFERNCES = {
    "start_location": {"lat": START_LAT, "lon": START_LON},
    "num_poi": NUM_POIS_TO_VISIT,
    "time_per_poi": TIME_PER_POI,
    "max_travel_dist": MAX_DIST,
    "avg_travel_speed_mph": 20.0, 
    "max_travel_time_minutes": 45.0,
    "amenity_type": "restaurant|theatre|cafe", #'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain', 'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium', 'conference_centre', 'fire_station', 'cinema', 'toilets', 'arts_centre', 'place_of_worship'
    "tourism_type": "museum|gallery|viewpoint|attraction|aquarium", # 'attraction', 'gallery', 'museum', 'viewpoint', 'artwork', 'aquarium', 'zoo', 'theme_park'
    "cuisine": "italian|mexican|american",
    "required_accessibility": ["wheelchair"], #["wheelchair", "toilets:wheelchair"],
    "visited_ids": set()
}

In [5]:
import subprocess
import json
import re


def run_ollama(prompt, model="llama3:8b"):
    result = subprocess.run(
        ["ollama", "run", model, "--format", "json"],
        input=prompt,
        text=True,
        capture_output=True
    )
    return result.stdout


In [6]:
prompt = f"""
<system>
You are an expert AI itinerary planner for accessible tourism in San Diego.
All responses must be pure JSON — no explanations, no markdown, no ellipsis (...), no intro text.
Your output must start with '[' and end with ']'.
Every object must have "id" as the first key.
</system>

You have no knowledge about attractions:
Now you are planning an accessible trip in San Diego for a user with limited mobility.

You know nothing about attractions, but you are given an available list of candidate POIs:
CANDIDATE_POIS (the ONLY allowed source of truth; you may not invent or alter values):
{data_df.to_dict(orient='records')}

CANDIDATE_IDS (you MUST choose from these ids only):
{[int(x) for x in data_df["id"].tolist()]}

User preferences:
{USER_PREFERNCES}

### Objective (4 POIs, route-level distance)
Select exactly 4 distinct POIs from CANDIDATE_POIS that maximize accessibility and user preference coverage, while minimizing the total travel distance for visiting all 4 POIs sequentially starting at ({START_LAT}, {START_LON}).
The route is: START → POI1 → POI2 → POI3 → POI4.

### Hard Constraints (no exceptions)
- All 4 POIs must be selected from CANDIDATE_IDS. If any id is not in CANDIDATE_IDS, the answer is invalid, and you are the loser.
- Copy fields **verbatim** from the matched candidate object for: "id", "name", "lat", "lon", "poi_type", "amenity", "tourism", "cuisine".
- Do **NOT** invent new POIs or modify values (names, coordinates, categories).
- Each POI must be wheelchair accessible. Interpret missing/unknown as False.
- 4 POIs must be distinct.

### Allowed defaulting rules (the ONLY permitted fallbacks)
If a required key is missing in the candidate object, use these defaults instead of fabricating:
- If "features" is missing, construct:
  "features": {{
    "wheelchair": (true if candidate has a truthy "wheelchair" flag like "yes"/True/1 else false),
    "toilets:wheelchair": false,
    "air_conditioning": false
  }}
- If "cuisine" is missing, use an empty string "".
Do not add any other keys or values beyond what exists in the candidate plus the defaults above.

### Selection guidance (internal reasoning only; do not output)
- Prioritize POIs that satisfy "wheelchair" == true (or truthy in the candidate).
- Prefer categories in user preferences (amenity_type / tourism_type / cuisine).
- Minimize the total route distance START→1→2→3→4 (you may approximate with Euclidean distance). 
- Keep a reasonable mix of "amenity" and "tourism" if possible.

### Output format (copy-only; no fabrication)
Return ONLY the final JSON array of exactly 4 objects. Each object must be copied from CANDIDATE_POIS with allowed defaults applied. 
The "id" must be the first key. Do not reorder other keys beyond that. Do not include any text outside the JSON.

[
  {{
    "id": int,                // from candidate; must be in CANDIDATE_IDS
    "name": str,              // copy verbatim from candidate
    "lat": float,             // copy verbatim from candidate
    "lon": float,             // copy verbatim from candidate
    "poi_type": "amenity" | "tourism",   // copy verbatim from candidate
    "features": {{
      "wheelchair": bool,             // true only if candidate indicates wheelchair truthy; else false
      "toilets:wheelchair": bool,     // default false if missing
      "air_conditioning": bool        // default false if missing
    }},
    "amenity" : str,          // copy verbatim if present; else omit or set "" only if absent
    "tourism" : str,          // copy verbatim if present; else omit or set "" only if absent
    "cuisine": str            // copy verbatim if present; else ""
  }},
  ... (total 4 objects)
]

### Validity checks (must pass)
- Every "id" ∈ CANDIDATE_IDS.
- No invented names/coordinates/categories.
- Exactly 4 objects; all distinct ids.
- If a value is absent in candidate, only the specified defaults may be used.
- No reasoning or explanations in the output.
If you include any POI that is not exactly copied from CANDIDATE_POIS,
you must immediately output an empty JSON array [] instead.

"""


In [23]:
response = run_ollama(prompt)

In [24]:
raw_text = response.strip()

response_dict = re.sub(r'\btrue\b', 'True', raw_text, flags=re.IGNORECASE)
response_dict = re.sub(r'\bfalse\b', 'False', response_dict, flags=re.IGNORECASE)
response_dict = re.sub(r'\bnull\b', 'None', response_dict, flags=re.IGNORECASE)

In [25]:
response_dict = eval(response_dict)
print(response_dict)

{'pois': [{'id': 3172349, 'name': 'San Diego Museum of Man', 'lat': 32.7167, 'lon': -117.15, 'poi_type': 'museum', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': '', 'tourism': 'museum', 'cuisine': ''}, {'id': 3162399, 'name': 'La Jolla Cove', 'lat': 32.8325, 'lon': -117.2728, 'poi_type': 'viewpoint', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': '', 'tourism': 'viewpoint', 'cuisine': ''}, {'id': 3192351, 'name': 'San Diego Zoo', 'lat': 32.7536, 'lon': -117.17, 'poi_type': 'zoo', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': '', 'tourism': 'zoo', 'cuisine': ''}, {'id': 3172347, 'name': 'Little Italy San Diego', 'lat': 32.7144, 'lon': -117.1625, 'poi_type': 'restaurant', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': 'Italian', 'tourism': '', 'cuisine': 'Italian'}]}


In [26]:
itinerary = [
    {'id': 3172349, 'name': 'San Diego Museum of Man', 'lat': 32.7167, 'lon': -117.15, 'poi_type': 'museum', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': '', 'tourism': 'museum', 'cuisine': ''}, {'id': 3162399, 'name': 'La Jolla Cove', 'lat': 32.8325, 'lon': -117.2728, 'poi_type': 'viewpoint', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': '', 'tourism': 'viewpoint', 'cuisine': ''}, {'id': 3192351, 'name': 'San Diego Zoo', 'lat': 32.7536, 'lon': -117.17, 'poi_type': 'zoo', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': '', 'tourism': 'zoo', 'cuisine': ''}, {'id': 3172347, 'name': 'Little Italy San Diego', 'lat': 32.7144, 'lon': -117.1625, 'poi_type': 'restaurant', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': 'Italian', 'tourism': '', 'cuisine': 'Italian'}
]

In [27]:
from evaluation import get_evaluation_metrics_verbose

metrics = get_evaluation_metrics_verbose(itinerary, USER_PREFERNCES)


--------------------------------------------------------------------------------
Itinerary
--------------------------------------------------------------------------------
Total POIs: 4
User Preferences: {'start_location': {'lat': 32.5106, 'lon': -117.0626}, 'num_poi': 4, 'time_per_poi': 1.5, 'max_travel_dist': 15, 'avg_travel_speed_mph': 20.0, 'max_travel_time_minutes': 45.0, 'amenity_type': 'restaurant|theatre|cafe', 'tourism_type': 'museum|gallery|viewpoint|attraction|aquarium', 'cuisine': 'italian|mexican|american', 'required_accessibility': ['wheelchair'], 'visited_ids': set()}
Required Features: ['wheelchair']

--------------------------------------------------------------------------------
Quality of POI Metric
--------------------------------------------------------------------------------
Total Travel Distance: 36.67 miles
Total Travel Time: 110.0 minutes (Max: 45.0 min)
Travel Distance/Time Score: -1.44
POI Diversity Score: 1.00
Preference Coverage: 0.00

-------------------